In [ ]:
from collections import defaultdict
from itertools import combinations

import kaldiio
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

from wespeaker.utils.score_metrics import (
    compute_pmiss_pfa_rbst,
    compute_eer,
    compute_c_norm,
)

# --- Configuration ---
embedding_scp = '/work/user_data/danwells/wespeaker/data-files/vctk/embeddings/voxceleb_ECAPA512.scp'
utt2spk_file = '/work/user_data/danwells/wespeaker/data-files/vctk/utt2spk'
MAX_TARGET_PAIRS = 50000
MAX_NONTARGET_PAIRS = 50000
SEED = 42

In [ ]:
# Load embeddings
emb_dict = {}
for utt, emb in kaldiio.load_scp_sequential(embedding_scp):
    emb_dict[utt] = emb

# Load utt2spk -> build spk2utts
spk2utts = defaultdict(list)
with open(utt2spk_file) as f:
    for line in f:
        utt_id, spk_id = line.strip().split(maxsplit=1)
        spk2utts[spk_id].append(utt_id)

# Filter to utterances present in embedding scp, drop speakers with <2 utts
spk2utts = {
    spk: [u for u in utts if u in emb_dict]
    for spk, utts in spk2utts.items()
}
spk2utts = {spk: utts for spk, utts in spk2utts.items() if len(utts) >= 2}

print(f"Loaded {len(emb_dict)} embeddings, "
      f"{len(spk2utts)} speakers with >=2 utts")

In [ ]:
rng = np.random.default_rng(SEED)

# Generate target pairs (same speaker)
target_pairs = []
for spk, utts in spk2utts.items():
    for u1, u2 in combinations(utts, 2):
        target_pairs.append((u1, u2))

# Generate non-target pairs (different speakers)
spk_list = list(spk2utts.keys())
num_spk = len(spk_list)
num_possible_nontarget = num_spk * (num_spk - 1) // 2

if num_possible_nontarget <= MAX_NONTARGET_PAIRS:
    # Enumerate all speaker pairs
    nontarget_pairs = []
    for i in range(num_spk):
        for j in range(i + 1, num_spk):
            u1 = rng.choice(spk2utts[spk_list[i]])
            u2 = rng.choice(spk2utts[spk_list[j]])
            nontarget_pairs.append((u1, u2))
else:
    # Sample speaker pairs directly
    nontarget_pairs = []
    seen = set()
    while len(nontarget_pairs) < MAX_NONTARGET_PAIRS:
        i, j = rng.choice(num_spk, size=2, replace=False)
        pair_key = (min(i, j), max(i, j))
        if pair_key in seen:
            continue
        seen.add(pair_key)
        u1 = rng.choice(spk2utts[spk_list[i]])
        u2 = rng.choice(spk2utts[spk_list[j]])
        nontarget_pairs.append((u1, u2))

# Subsample target pairs if needed
if len(target_pairs) > MAX_TARGET_PAIRS:
    idx = rng.choice(len(target_pairs), MAX_TARGET_PAIRS, replace=False)
    target_pairs = [target_pairs[i] for i in idx]
if len(nontarget_pairs) > MAX_NONTARGET_PAIRS:
    idx = rng.choice(len(nontarget_pairs), MAX_NONTARGET_PAIRS, replace=False)
    nontarget_pairs = [nontarget_pairs[i] for i in idx]

print(f"Target pairs: {len(target_pairs)}, "
      f"Non-target pairs: {len(nontarget_pairs)}")

# Score all pairs (vectorized cosine similarity)
def score_pairs(pairs):
    emb1 = np.stack([emb_dict[p[0]] for p in pairs])
    emb2 = np.stack([emb_dict[p[1]] for p in pairs])
    num = np.sum(emb1 * emb2, axis=1)
    denom = np.linalg.norm(emb1, axis=1) * np.linalg.norm(emb2, axis=1)
    return num / denom

target_scores = score_pairs(target_pairs)
nontarget_scores = score_pairs(nontarget_pairs)

scores = np.concatenate([target_scores, nontarget_scores])
labels = np.concatenate([np.ones(len(target_scores)),
                         np.zeros(len(nontarget_scores))])

In [ ]:
fnr, fpr = compute_pmiss_pfa_rbst(scores, labels)
eer, threshold = compute_eer(fnr, fpr, scores)

min_dcf_08 = compute_c_norm(fnr, fpr, p_target=0.01, c_miss=1, c_fa=1)
min_dcf_10 = compute_c_norm(fnr, fpr, p_target=0.001, c_miss=1, c_fa=1)

print(f"EER: {eer*100:.2f}% (threshold: {threshold:.4f})")
print(f"minDCF(p=0.01): {min_dcf_08:.4f}")
print(f"minDCF(p=0.001): {min_dcf_10:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Score histogram
ax = axes[0]
ax.hist(nontarget_scores, bins=100, alpha=0.6, density=True,
        label='Non-target', color='blue')
ax.hist(target_scores, bins=100, alpha=0.6, density=True,
        label='Target', color='red')
ax.axvline(threshold, color='k', linestyle='--',
           label=f'EER threshold ({threshold:.3f})')
ax.set_xlabel('Cosine similarity score')
ax.set_ylabel('Density')
ax.set_title('Score Distribution')
ax.legend()

# DET curve
ax = axes[1]
p_miss = norm.ppf(np.clip(fnr, 1e-6, 1 - 1e-6))
p_fa = norm.ppf(np.clip(fpr, 1e-6, 1 - 1e-6))
ax.plot(p_fa, p_miss, 'r')

xytick = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.4]
xytick_labels = [str(x * 100) for x in xytick]
ax.set_xticks(norm.ppf(xytick))
ax.set_xticklabels(xytick_labels)
ax.set_yticks(norm.ppf(xytick))
ax.set_yticklabels(xytick_labels)
ax.set_xlim(norm.ppf([0.001, 0.5]))
ax.set_ylim(norm.ppf([0.001, 0.5]))
ax.set_xlabel('False alarm rate [%]')
ax.set_ylabel('False reject rate [%]')
ax.set_title('DET Curve')
ax.plot(norm.ppf(eer), norm.ppf(eer), 'ko', markersize=8)
ax.annotate(f'EER = {eer*100:.2f}%',
            xy=(norm.ppf(eer), norm.ppf(eer)),
            xytext=(norm.ppf(eer + 0.001), norm.ppf(eer + 0.001)),
            fontsize=10,
            #arrowprops=dict(arrowstyle='-|>',
            #                connectionstyle='arc3,rad=0.2', fc='w'),
            #bbox=dict(boxstyle='round4', fc='w')
           )
ax.grid(True)

plt.tight_layout()
plt.show()